In [2]:
LAST_NAME = "Papa"
STUDENT_ID = "TUPM-26-1927"
FAVORITE_ARTIST = "Maki"

id_digits = [int(c) for c in STUDENT_ID if c.isdigit()]
letters = [c for c in FAVORITE_ARTIST.upper() if c.isalpha()]

SEED_NUM = id_digits[-1]    # CHECK this against the SEED_NUM rule in your Lab 4 handout

print("SEED_NUM        :", SEED_NUM)
print("LAST_NAME       :", LAST_NAME)
print("FAVORITE_ARTIST :", FAVORITE_ARTIST)


SEED_NUM        : 7
LAST_NAME       : Papa
FAVORITE_ARTIST : Maki


In [3]:
sources = [f"{LAST_NAME}-T{i}" for i in range(1, 6)]
categories = ["TEMPERATURE", "VOLTAGE", "CURRENT", "PRESSURE", "SPEED"]

counter = 0
def next_value():
    """Value from SEED_NUM and the next character position of FAVORITE_ARTIST."""
    global counter
    pos = counter % len(letters) + 1                    # character position (1-based)
    letter_value = ord(letters[pos - 1]) - ord("A") + 1  # A=1 ... Z=26
    counter += 1
    return letter_value * pos + SEED_NUM

records = []

# one record per source, each with a different category
for i in range(5):
    records.append({"source": sources[i], "category": categories[i], "value": next_value()})

# extra measurements: TEMPERATURE, VOLTAGE and CURRENT get multiple readings
for s, cat in [(0, "TEMPERATURE"), (1, "TEMPERATURE"), (1, "VOLTAGE"), (2, "VOLTAGE"), (3, "CURRENT")]:
    records.append({"source": sources[s], "category": cat, "value": next_value()})

# one repeated record (exact copy of the first record)
records.append(dict(records[0]))

print("Sources   :", sources)
print("Categories:", categories)
print()
print(f"{'#':<4}{'Source':<12}{'Category':<14}{'Value':>6}")
for n, r in enumerate(records, 1):
    print(f"{n:<4}{r['source']:<12}{r['category']:<14}{r['value']:>6}")

Sources   : ['Papa-T1', 'Papa-T2', 'Papa-T3', 'Papa-T4', 'Papa-T5']
Categories: ['TEMPERATURE', 'VOLTAGE', 'CURRENT', 'PRESSURE', 'SPEED']

#   Source      Category       Value
1   Papa-T1     TEMPERATURE       20
2   Papa-T2     VOLTAGE            9
3   Papa-T3     CURRENT           40
4   Papa-T4     PRESSURE          43
5   Papa-T5     SPEED             20
6   Papa-T1     TEMPERATURE        9
7   Papa-T2     TEMPERATURE       40
8   Papa-T2     VOLTAGE           43
9   Papa-T3     VOLTAGE           20
10  Papa-T4     CURRENT            9
11  Papa-T1     TEMPERATURE       20


In [4]:
organized = {}
for r in records:
    organized.setdefault(r["source"], {}).setdefault(r["category"], []).append(r["value"])

for src, cats in organized.items():
    print(src)
    for cat, vals in cats.items():
        print(f"   {cat:<12}: {vals}")

Papa-T1
   TEMPERATURE : [20, 9, 20]
Papa-T2
   VOLTAGE     : [9, 43]
   TEMPERATURE : [40]
Papa-T3
   CURRENT     : [40]
   VOLTAGE     : [20]
Papa-T4
   PRESSURE    : [43]
   CURRENT     : [9]
Papa-T5
   SPEED       : [20]


In [5]:
from collections import Counter

counts = Counter((r["source"], r["category"], r["value"]) for r in records)
repeated = [k for k, c in counts.items() if c > 1]
distinct = set(counts)

print("Total records   :", len(records))
print("Distinct records:", len(distinct))
print("Repeated records:", repeated)
print()
print("Distinct sources   :", sorted({r["source"] for r in records}))
print("Distinct categories:", sorted({r["category"] for r in records}))
print("Distinct values    :", sorted({r["value"] for r in records}))

Total records   : 11
Distinct records: 10
Repeated records: [('Papa-T1', 'TEMPERATURE', 20)]

Distinct sources   : ['Papa-T1', 'Papa-T2', 'Papa-T3', 'Papa-T4', 'Papa-T5']
Distinct categories: ['CURRENT', 'PRESSURE', 'SPEED', 'TEMPERATURE', 'VOLTAGE']
Distinct values    : [9, 20, 40, 43]


In [6]:
by_category = {}
for r in records:
    by_category.setdefault(r["category"], []).append(r["value"])

cat_summary = {}
print(f"{'Category':<14}{'Count':>6}{'Min':>6}{'Max':>6}{'Sum':>6}{'Avg':>8}")
for cat, vals in by_category.items():
    cat_summary[cat] = {"count": len(vals), "min": min(vals), "max": max(vals),
                        "sum": sum(vals), "avg": sum(vals) / len(vals)}
    s = cat_summary[cat]
    print(f"{cat:<14}{s['count']:>6}{s['min']:>6}{s['max']:>6}{s['sum']:>6}{s['avg']:>8.2f}")

print()
print("Records per source:", dict(Counter(r["source"] for r in records)))
all_values = [r["value"] for r in records]
print("Overall min / max / avg:", min(all_values), max(all_values), round(sum(all_values) / len(all_values), 2))

Category       Count   Min   Max   Sum     Avg
TEMPERATURE        4     9    40    89   22.25
VOLTAGE            3     9    43    72   24.00
CURRENT            2     9    40    49   24.50
PRESSURE           1    43    43    43   43.00
SPEED              1    20    20    20   20.00

Records per source: {'Papa-T1': 3, 'Papa-T2': 3, 'Papa-T3': 2, 'Papa-T4': 2, 'Papa-T5': 1}
Overall min / max / avg: 9 43 24.82


In [7]:
processed = {(src, cat): sum(v) / len(v)
             for src, cats in organized.items() for cat, v in cats.items()}

print("Processed results (average per source and category):")
for (src, cat), avg in processed.items():
    print(f"  {src:<12}{cat:<14}{avg:>8.2f}")

print("\nOrdered results (highest average first):")
for rank, ((src, cat), avg) in enumerate(sorted(processed.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"  {rank}. {src:<12}{cat:<14}{avg:>8.2f}")

print("\nRecords ordered by source, category, value:")
for r in sorted(records, key=lambda r: (r["source"], r["category"], r["value"])):
    print(f"  {r['source']:<12}{r['category']:<14}{r['value']:>6}")

Processed results (average per source and category):
  Papa-T1     TEMPERATURE      16.33
  Papa-T2     VOLTAGE          26.00
  Papa-T2     TEMPERATURE      40.00
  Papa-T3     CURRENT          40.00
  Papa-T3     VOLTAGE          20.00
  Papa-T4     PRESSURE         43.00
  Papa-T4     CURRENT           9.00
  Papa-T5     SPEED            20.00

Ordered results (highest average first):
  1. Papa-T4     PRESSURE         43.00
  2. Papa-T2     TEMPERATURE      40.00
  3. Papa-T3     CURRENT          40.00
  4. Papa-T2     VOLTAGE          26.00
  5. Papa-T3     VOLTAGE          20.00
  6. Papa-T5     SPEED            20.00
  7. Papa-T1     TEMPERATURE      16.33
  8. Papa-T4     CURRENT           9.00

Records ordered by source, category, value:
  Papa-T1     TEMPERATURE        9
  Papa-T1     TEMPERATURE       20
  Papa-T1     TEMPERATURE       20
  Papa-T2     TEMPERATURE       40
  Papa-T2     VOLTAGE            9
  Papa-T2     VOLTAGE           43
  Papa-T3     CURRENT           40

In [8]:
highest = max(cat_summary, key=lambda c: cat_summary[c]["avg"])
lowest = min(cat_summary, key=lambda c: cat_summary[c]["avg"])

print("=== FINAL ORGANIZED AND SUMMARIZED TEST DATASET ===")
print("Total records     :", len(records))
print("Distinct records  :", len(distinct))
print("Repeated records  :", len(records) - len(distinct))
print("Highest avg category:", highest, "->", round(cat_summary[highest]["avg"], 2))
print("Lowest avg category :", lowest, "->", round(cat_summary[lowest]["avg"], 2))

=== FINAL ORGANIZED AND SUMMARIZED TEST DATASET ===
Total records     : 11
Distinct records  : 10
Repeated records  : 1
Highest avg category: PRESSURE -> 43.0
Lowest avg category : SPEED -> 20.0
